In [1]:
import os
from  pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text
from gspread.utils import rowcol_to_a1
from gspread_dataframe import set_with_dataframe

# Установка базовой директории и пути к файлу с учетными данными. Используем конструкцию try-except для обработки возможных ошибок при определении пути для notebook.
try:
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    BASE_DIR = Path.cwd().resolve().parents[1] 

SRC_DIR = os.path.join(BASE_DIR, 'src')

# Добавляем src в пути поиска модулей
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from core.google_sheets_scheme import week_n_redeem
from core.utils_gspread import safe_open_spreadsheet
from core.utils_sql import get_db_engine

In [23]:
def update_week_n_redeem():
	# Получаем подключение к базе данных
	engine = get_db_engine()
	# Выполняем SQL-запрос для получения данных по еженедельным отчетам реализации и уведомлениях о выпкупе
	query = text("""-- === Для бухгалтерии ===
WITH week_rep AS ( -- Данные еженедельного отчета реализации
	SELECT 
		-- Достаю данные по различным статьям еженедельного отчета в разрезе документа
		SUM(
		CASE 
			WHEN w.title = 'Итого стоимость реализованного товара и услуг'
				THEN COALESCE(sum_rub, 0)
		END) AS total_sum,
		ROUND(SUM(
		    CASE 
		        WHEN w.title = 'Итого стоимость реализованного товара и услуг'
		            THEN COALESCE(sum_rub, 0) / (1.0 + v.vat)  
		    END
		), 2) AS total_sum_without_vat,
			SUM(
		CASE 
			WHEN w.title IN ('Компенсация ущерба', 'Прочие выплаты')
				THEN COALESCE(sum_rub, 0)
		END) AS damages_comp_other,
			SUM(
		CASE 
			WHEN w.title = 'Компенсация скидки по программе лояльности'
				THEN COALESCE(sum_rub, 0)
		END) AS discount_loyalty,
		SUM(
		CASE 
			WHEN w.title IN ('Сумма вознаграждения Вайлдберриз за текущий период (ВВ), без НДС', 'НДС с вознаграждения Вайлдберриз')
				THEN COALESCE(sum_rub, 0)
		END) AS award,
		SUM(
		CASE 
			WHEN w.title IN ('Сумма, удержанная в счёт обеспечения организации платежа')
				THEN COALESCE(sum_rub, 0)
		END) AS amount_withheld_to_org,
		SUM(
		CASE 
			WHEN w.title IN ('Возмещение расходов по перевозке')
				THEN COALESCE(sum_rub, 0)
		END) AS reimbursement_of_transp_costs,
		SUM(
		CASE 
			WHEN w.title IN ('Возмещение за выдачу и возврат товаров на ПВЗ')
				THEN COALESCE(sum_rub, 0)
		END) AS reimbursement_for_delivery_and_return_of_goods_to_pvz,	
		SUM(
		CASE 
			WHEN w.title IN ('Штрафы')
				THEN COALESCE(sum_rub, 0)
		END) AS penalties,
		SUM(
		CASE 
			WHEN w.title IN ('Прочие удержания')
				THEN COALESCE(sum_rub, 0)
		END) other_deductions,	
		SUM(
		CASE 
			WHEN w.title IN ('Удержания в пользу третьих лиц')
				THEN COALESCE(sum_rub, 0)
		END) retentions_in_favor_of_third_parties,	
		-- Выделяю данные для группировки
		w.doc_num AS weekly_rep,
		DATE(DATE_TRUNC('week', w."date")) AS week_start,
		DATE(w."date") AS report_date,
		DATE((DATE_TRUNC('week', w."date") + INTERVAL '7 days' - INTERVAL '1 microsecond')) AS week_end,
		w.account
	FROM weekly_implementation_report w
	LEFT JOIN vat_guide v
		ON v.account = UPPER(w.account)
	GROUP BY w.doc_num, w."date", w.account),
	fin_rep AS -- Данные еженедельного финансового отчета
		(SELECT 
			f.realizationreport_id,
			f.date_from,
			f.date_to,
			f.create_dt,
			sum
			(CASE
				WHEN f.doc_type_name ILIKE '%возврат%'
				THEN (f.ppvz_for_pay)
				ELSE 0
			END) AS return_pay,
			f.account
		FROM fin_reports_full f
		WHERE f.report_type = 2
		GROUP BY f.realizationreport_id, f.date_from, f.date_to, f.create_dt, f.account),
	redeem_not AS( -- Данные уведомления о выкупе
		SELECT sum(sum_rub_with_vat) AS sum_rub_with_vat,
		-- Считаем в зависимости от ставки НДС
		ROUND(sum(sum_rub_with_vat)/(1.0 + v.vat), 2) AS sum_rub_without_vat,
		SUBSTRING(r.doc_name FROM '№(\d+)') AS redeem_notif, -- из полного названия документа извлекаю номер
		DATE(SUBSTRING(r.doc_name FROM ' от (\d{4}-\d{2}-\d{2})')) AS redeem_notif_date
	FROM redeem_notification r
	LEFT JOIN vat_guide v
		ON v.account = UPPER(r.account)
	GROUP BY r.doc_name, v.vat)
	SELECT 
		w.account,
		w.weekly_rep AS "Номер_еженедельного_отчета",
		w.week_start AS "Начало_периода",
		w.report_date AS "Конец_периода",
		w.total_sum AS "Всего_товара",
		w.total_sum_without_vat AS "Всего_товара_БЕЗ_НДС",
		w.damages_comp_other AS "Компенсации_ущерба_и_прочие_выплаты",
		w.discount_loyalty AS "Компенсация_скидки_по_программе_лояльности",
		r.redeem_notif AS "Уведомление_о_выкупе_№",
		r.sum_rub_with_vat AS "Выкуплено_по_уведомлению",
		r.sum_rub_without_vat AS "Выкуплено_по_уведомлению_без_НДС",
		f.return_pay AS "Вовзрат_выкупа",
		CASE 
			WHEN w.award < 0
			THEN w.award*-1
			ELSE w.award 
		END AS "Вознагрождение_в_доход",
		CASE 
			WHEN w.award < 0
			-- Расчет в зависимости от налоговой ставки
				THEN ROUND((w.award*-1)/(1.0 + v.vat), 2)
			ELSE 0 
		END AS "Вознагрождение_в_доход_БЕЗ_НДС",	
		w.award AS "Вознаграждение",
		COALESCE(amount_withheld_to_org, 0) AS "Сумма_удержанная_в_счёт_обеспечения_организации_платежа",
		reimbursement_of_transp_costs AS "Возмещение расходов по перевозке",
		reimbursement_for_delivery_and_return_of_goods_to_pvz AS "Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ", 
		penalties AS "Штрафы",
		other_deductions AS "Прочие удержания",
		retentions_in_favor_of_third_parties AS "Удержания_в_пользу_третьих_лиц"
	FROM week_rep w
	LEFT JOIN fin_rep f ON UPPER(w.account) = UPPER(f.account) 
		AND w.report_date = f.date_to
	LEFT JOIN redeem_not r
		ON f.realizationreport_id = r.redeem_notif::INT
	LEFT JOIN vat_guide v
		ON v.account = UPPER(w.account)
	WHERE f.date_to > '2025-12-31'
	ORDER BY w.account, w.week_start DESC;""")

	# Загружаем данные в DataFrame
	df = pd.read_sql(query, engine)
	df = df.rename(columns={
							"Всего_товара":             "Всего_стоимость_реализованного_товара",
							"Всего_товара_БЕЗ_НДС": "Всего_стоимость_реализованного_товара_без_НДС",
							"Компенсации_ущерба_и_прочие_выпла": "Компенсации_ущерба_и_прочие_выплаты",
							"Компенсация_скидки_по_программе_л": "Компенсация_скидки_по_программе_лояльности",
							"Возмещение_за_выдачу_и_возврат_тов": "Возмещение_за_выдачу_и_возврат_товара_ПВЗ",
							"Сумма_удержанная_в_счёт_обеспечен":"Сумма_удержанная_в_счёт_обеспечения_организации_платежа",
							"Возмещение_за_выдачу_и_возврат_тов":"Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ"
							})
	# Сортировка по неделям и аккаунтам
	df = df.sort_values(by=['Начало_периода', 'account'], ascending=[False, True])
	return df
	# # Открываем таблицу 
	# report_table = safe_open_spreadsheet(week_n_redeem['title'])
	# # Обновляем данные гугл таблицы
	# report_sheet = report_table.worksheet(week_n_redeem['data'])
	# set_with_dataframe(report_sheet, df)

<>:95: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:95: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\123\AppData\Local\Temp\ipykernel_1364\1205972970.py:95: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  SUBSTRING(r.doc_name FROM '№(\d+)') AS redeem_notif, -- из полного названия документа извлекаю номер


In [24]:
df = update_week_n_redeem()

In [9]:
df.columns

Index(['account', 'Номер_еженедельного_отчета', 'Начало_периода',
       'Конец_периода', 'Всего_стоимость_реализованного_товара',
       'Всего_стоимость_реализованного_товара_без_НДС',
       'Компенсации_ущерба_и_прочие_выплаты',
       'Компенсация_скидки_по_программе_лояльности', 'Уведомление_о_выкупе_№',
       'Выкуплено_по_уведомлению', 'Выкуплено_по_уведомлению_без_НДС',
       'Вовзрат_выкупа', 'Вознагрождение_в_доход',
       'Вознагрождение_в_доход_БЕЗ_НДС', 'Вознаграждение',
       'Сумма_удержанная_в_счёт_обеспечения_организации_платежа',
       'Возмещение расходов по перевозке',
       'Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ', 'Штрафы',
       'Прочие удержания', 'Удержания_в_пользу_третьих_лиц', 'Итого_расходы'],
      dtype='object')

In [ ]:
import numpy as np

df['Итого_расходы'] = np.where(
    df['Вознаграждение'] < 0,
    df['Вовзрат_выкупа'] + 
    df['Возмещение расходов по перевозке'] +
    df['Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ'] + 
    df['Штрафы'] + 
    df['Прочие удержания'] + 
    df['Удержания_в_пользу_третьих_лиц'] + 
    df['Сумма_удержанная_в_счёт_обеспечения_организации_платежа'],
    # Иначе (ВОЗНАГРАЖДЕНИЕ >= 0):
    df['Вовзрат_выкупа'] + 
    df['Возмещение расходов по перевозке'] +
    df['Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ'] + 
    df['Штрафы'] + 
    df['Прочие удержания'] + 
    df['Удержания_в_пользу_третьих_лиц'] + 
    df['Сумма_удержанная_в_счёт_обеспечения_организации_платежа'] +
    df['Вознаграждение'],

)


df['К_перечислению_по_отчетам'] = np.where(
    df['Вознаграждение'] >= 0,
    # Если вознаграждение >= 0
    df['Всего_стоимость_реализованного_товара'] +
    df['Компенсации_ущерба_и_прочие_выплаты'] + 
    df['Выкуплено_по_уведомлению'] +
    df['Вознаграждение'] - 
    df['Итого_расходы'],
    # Если вознаграждение < 0
    df['Всего_стоимость_реализованного_товара'] +
    df['Компенсации_ущерба_и_прочие_выплаты'] + 
    df['Выкуплено_по_уведомлению'] -
    df['Вознаграждение'] - 
    df['Итого_расходы']

)
df.head()

,account,Номер_еженедельного_отчета,Начало_периода,Конец_периода,Всего_стоимость_реализованного_товара,Всего_стоимость_реализованного_товара_без_НДС,Компенсации_ущерба_и_прочие_выплаты,Компенсация_скидки_по_программе_лояльности,Уведомление_о_выкупе_№,Выкуплено_по_уведомлению,...,Вознагрождение_в_доход_БЕЗ_НДС,Вознаграждение,Сумма_удержанная_в_счёт_обеспечения_организации_платежа,Возмещение расходов по перевозке,Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ,Штрафы,Прочие удержания,Удержания_в_пользу_третьих_лиц,Итого_расходы,К_перечислению_по_отчетам
0,ВЕКТОР,641131933.0,2026-02-23,2026-03-01,8489045.44,6958233.97,422833.63,8650.35,641131935,720993.08,...,2159430.66,-2634505.41,315549.06,514054.75,229946.53,132463.62,1358575.0,1281489.08,3836685.03,8430692.53
33,ДАНИЕЛЯН,640938673.0,2026-02-23,2026-03-01,2890911.01,2701785.99,82.85,3822.94,640938674,208104.61,...,951465.50,-1018068.08,108776.97,189881.76,79835.15,1056.35,486943.0,142481.61,1010166.98,3106999.57
42,ЛОПАТИНА,641078376.0,2026-02-23,2026-03-01,894128.85,835634.44,1780.46,569.84,641078383,77856.25,...,192772.80,-206266.90,31905.84,42697.40,23197.72,16326.00,183895.0,40394.00,338415.96,841616.50
51,ОГАНЕСЯН,640998014.0,2026-02-23,2026-03-01,2208765.30,2064266.64,4552.87,1864.51,640998026,167704.80,...,622068.10,-665612.87,84041.45,129216.05,57897.91,0.00,468732.0,138410.69,880380.40,2166255.44
60,ПИЛОСЯН,641844699.0,2026-02-23,2026-03-01,6323829.10,5910120.65,0.00,3554.31,641844700,501367.35,...,1697882.64,-1816734.42,230564.35,346631.99,164909.36,18158.67,1324348.0,142774.17,2234069.49,6407861.38


In [47]:
df[df['Номер_еженедельного_отчета'] == 614397634]

,account,Номер_еженедельного_отчета,Начало_периода,Конец_периода,Всего_стоимость_реализованного_товара,Всего_стоимость_реализованного_товара_без_НДС,Компенсации_ущерба_и_прочие_выплаты,Компенсация_скидки_по_программе_лояльности,Уведомление_о_выкупе_№,Выкуплено_по_уведомлению,...,Вознагрождение_в_доход_БЕЗ_НДС,Вознаграждение,Сумма_удержанная_в_счёт_обеспечения_организации_платежа,Возмещение расходов по перевозке,Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ,Штрафы,Прочие удержания,Удержания_в_пользу_третьих_лиц,Итого_расходы,К_перечислению_по_отчетам
103,СТАРТ9237,614397634.0,2026-01-26,2026-02-01,91398.0,74916.39,0.0,0.0,616866901,10134.91,...,0.0,31659.23,2382.89,7078.17,2376.37,0.02,0.0,0.0,43496.68,89695.46
